# Creation and Application of Knowledge Graphs – Introduction

## Contents

### Python Libraries: RDFLib & SPARQLWrapper
- [RDFLib](https://rdflib.readthedocs.io/): Python package for working with RDF
  - Example 1: Create and Query a Graph
- [SPARQLWrapper](https://sparqlwrapper.readthedocs.io/): Simple Python wrapper around a SPARQL service
  - Examlpe 2: Query DBpedia with SPARQLWrapper

In [ ]:
!pip install -q rdflib SPARQLWrapper

## Example 1: Create and Query a Graph

In [ ]:
from rdflib import Graph, Namespace, Literal, URIRef
from rdflib.namespace import RDF, RDFS

### Create a Graph

Create a new graph and assign namespaces to it.

In [ ]:
#Creating a new Grpah
G = Graph()

# Required Namespaces
DBO = Namespace("http://dbpedia.org/ontology/")

# add the namespace to the graph
G.bind("dbo", DBO)

### Add Triples to the Graph

Let's consider two persons, Anna and Michael. Both of them belongs to the class 'Person'. First we create URI references.

In [ ]:
person1 = URIRef("http://example.org/anna")
person2 = URIRef("http://example.org/Michael")

# Adding nodes to graph
G.add((person1, RDF.type, DBO.Person))
G.add((person1, RDFS.label, Literal("Anna")))

G.add((person2, RDF.type, DBO.Person))
G.add((person2, RDFS.label, Literal("Michael")))

Add a triple expressing that Michael is the child of Anna.

In [ ]:
G.add((person1, DBO.child, person2))
print("Number of triples in G:", len(G))

### Visualize the Graph

In [ ]:
!pip install -q pydot

In [ ]:
import io
import pydot
from IPython.display import Image, display
from rdflib.tools.rdf2dot import rdf2dot

def visualize(g):
    stream = io.StringIO()
    rdf2dot(g, stream)
    graphs = pydot.graph_from_dot_data(stream.getvalue())
    display(Image(graphs[0].create_png()))

visualize(G)

### Save the Graph

Serialise and save the Graph for reuse. The graphs can be formatted in JSON, RDF/XML or Turtle syntax

In [ ]:
G.serialize(destination="output.ttl", format="turtle")

### Load the Saved Graph & Print the Triples

In [ ]:
G2 = Graph()
G2.parse("output.ttl", format="ttl")

for s, p ,o in G2:
    print(f'{s} -> {p} -> {o}')

In [ ]:
nm = G2.namespace_manager
G2.bind("ex", Namespace("http://example.org/"))
for s, p, o in G2:
    print(f"{s.n3(nm)} -> {p.n3(nm)} -> {o.n3(nm)}")

In [ ]:
for s, p, o in sorted(G2):
    print(f"{s.n3(nm)} -> {p.n3(nm)} -> {o.n3(nm)}")

In [ ]:
print(G2.serialize(format="turtle"))

### Query the Graph


In [ ]:
q = """
    PREFIX dbo: <http://dbpedia.org/ontology/>

    SELECT ?label
    WHERE {
        ?p rdf:type dbo:Person .
        ?p rdfs:label ?label .
    }
"""

results = G2.query(q)
print(results)

for row in results:
  print(row.label)



In [ ]:
print(results.serialize(format="csv").decode())

## Example 2: Query DBpedia with SPARQLWrapper

We can query the DBpedia knowledge graph using its [SPARQL endpoint](https://dbpedia.org/sparql).
To query it in Python code, we can use the SPARQLWrapper library.

Example query to find the spouse(s) of Donald Trump and their birth dates:

```
PREFIX dbr: <http://dbpedia.org/resource/>
PREFIX dbo: <http://dbpedia.org/ontology/>

SELECT ?spouse ?birthDate WHERE {
    dbr:Donald_Trump dbo:spouse ?spouse .
    ?spouse dbo:birthDate ?birthDate .
}
```

In [ ]:
from SPARQLWrapper import SPARQLWrapper, JSON

sparql = SPARQLWrapper(
    "https://dbpedia.org/sparql"
)
sparql.setReturnFormat(JSON)

sparql.setQuery("""
    PREFIX dbr: <http://dbpedia.org/resource/>
    PREFIX dbo: <http://dbpedia.org/ontology/>

    SELECT ?spouse ?birthDate WHERE {
        dbr:Donald_Trump dbo:spouse ?spouse .
        ?spouse dbo:birthDate ?birthDate .
    }
    """
)

try:
    ret = sparql.queryAndConvert()

    for row in ret["results"]["bindings"]:
        print(row)
except Exception as e:
    print(e)

In [ ]:
for row in ret["results"]["bindings"]:
    print(row["spouse"]["value"], "->", row["birthDate"]["value"])